In [17]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from sklearn.feature_selection import SelectFpr, SelectKBest, chi2, VarianceThreshold, mutual_info_classif, mutual_info_regression, RFECV
from sklearn.linear_model import LogisticRegression, Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.utils import resample
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from boruta import BorutaPy
warnings.filterwarnings('ignore')

k_features = 11

In [18]:
def fs_correlation(x, y, threshold=0.7):
    col_corr = set()
    corr_matrix = x.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) >= threshold:
                col_corr.add(corr_matrix.columns[i])
    return [c for c in x.columns if c not in col_corr]

In [19]:
def fs_chi_square(x, y):
    selector = SelectFpr(score_func=chi2, alpha=0.05)
    selector.fit(x, y)
    cols = list(x.columns[selector.get_support()])
    if len(cols) == 0:
        selector = SelectKBest(score_func=chi2, k=k_features).fit(x, y)
        cols = list(x.columns[selector.get_support()])
    return cols

In [20]:
def fs_variance(x, y, threshold=0.1):
    var = VarianceThreshold(threshold=threshold)
    var = var.fit(x)
    cols = list(x.columns[var.get_support(indices=True)])
    if len(cols) == 0:
        cols = list(x.var().sort_values(ascending=False).index[:k_features])
    return cols

In [21]:
def fs_mann_whitney(x, y):
    pvalues = {}
    for c in x.columns:
        a = x.loc[y == 0, c]
        b = x.loc[y == 1, c]
        try:
            pvalues[c] = mannwhitneyu(a, b).pvalue
        except ValueError:
            pvalues[c] = 1.0
    pvalues = pd.Series(pvalues)
    cols = list(pvalues[pvalues < 0.05].index)
    if len(cols) == 0:
        cols = list(pvalues.sort_values().index[:k_features])
    return cols

In [22]:
def fisher_score(x, y):
    yv = np.asarray(y)
    mu = x.mean().values
    num = np.zeros(x.shape[1])
    den = np.zeros(x.shape[1])
    for c in np.unique(yv):
        xc = x[yv == c]
        nc = len(xc)
        num += nc * (xc.mean().values - mu) ** 2
        den += nc * xc.var(ddof=0).values
    score = num / np.where(den == 0, np.nan, den)
    return pd.Series(score, index=x.columns).fillna(0)

def fs_fisher(x, y):
    return list(fisher_score(x, y).sort_values(ascending=False).index[:k_features])

In [23]:
def entropy(labels):
    p = np.bincount(labels) / len(labels)
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

def information_gain(x, y, bins=10):
    yv = np.asarray(y).astype(int)
    h = entropy(yv)
    gains = []
    for c in x.columns:
        col = x[c].values
        if len(np.unique(col)) > bins:
            col = pd.qcut(pd.Series(col), q=bins, duplicates='drop', labels=False).values
        cond = 0
        for v in np.unique(col):
            m = col == v
            cond += m.mean() * entropy(yv[m])
        gains.append(h - cond)
    return pd.Series(gains, index=x.columns)

def fs_information_gain(x, y):
    return list(information_gain(x, y).sort_values(ascending=False).index[:k_features])

In [24]:
def relieff(x, y, n_neighbors=10):
    X = x.values.astype(float)
    yv = np.asarray(y)
    n, p = X.shape
    span = X.max(0) - X.min(0)
    span[span == 0] = 1
    Z = X / span
    d = np.abs(Z[:, None, :] - Z[None, :, :]).sum(2)
    np.fill_diagonal(d, np.inf)
    w = np.zeros(p)
    for i in range(n):
        same = np.where(yv == yv[i])[0]
        same = same[same != i]
        diff = np.where(yv != yv[i])[0]
        hits = same[np.argsort(d[i, same])[:min(n_neighbors, len(same))]]
        miss = diff[np.argsort(d[i, diff])[:min(n_neighbors, len(diff))]]
        w += np.abs(Z[i] - Z[miss]).mean(0) - np.abs(Z[i] - Z[hits]).mean(0)
    return pd.Series(w / n, index=x.columns)

def fs_relief(x, y):
    return list(relieff(x, y, n_neighbors=1).sort_values(ascending=False).index[:k_features])

def fs_relieff(x, y):
    return list(relieff(x, y, n_neighbors=10).sort_values(ascending=False).index[:k_features])

In [25]:
def fs_mrmr(x, y, k=k_features):
    discrete = [x[c].nunique() <= 3 for c in x.columns]
    relevance = pd.Series(mutual_info_classif(x, y, discrete_features=discrete, random_state=42), index=x.columns)
    selected = [relevance.idxmax()]
    remaining = [c for c in x.columns if c != selected[0]]
    while len(selected) < k and len(remaining) > 0:
        scores = {}
        for c in remaining:
            redundancy = np.mean([mutual_info_regression(x[[s]], x[c], random_state=42)[0] for s in selected])
            scores[c] = relevance[c] - redundancy
        best = max(scores, key=scores.get)
        selected.append(best)
        remaining.remove(best)
    return selected

In [26]:
def fs_mutual_information(x, y):
    discrete = [x[c].nunique() <= 3 for c in x.columns]
    mi = pd.Series(mutual_info_classif(x, y, discrete_features=discrete, random_state=42), index=x.columns)
    return list(mi.sort_values(ascending=False).index[:k_features])

In [27]:
def get_estimators(kind):
    return {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'Logistic Regression': LogisticRegression(max_iter=2000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42, n_jobs=-1),
        'CatBoost': CatBoostClassifier(iterations=100, verbose=0, allow_writing_files=False, random_state=42, thread_count=-1),
        'AdaBoost': AdaBoostClassifier(random_state=42),
        'LightGBM': LGBMClassifier(n_estimators=100, verbose=-1, random_state=42, n_jobs=-1),
        'Support Vector Machine': SVC(kernel='linear') if kind == 'rfecv' else SVC(kernel='rbf'),
        'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(random_state=42)
    }

def fs_rfecv(x, y, name):
    estimator = get_estimators('rfecv')[name]
    selector = RFECV(estimator, step=1, cv=5, scoring='roc_auc', min_features_to_select=4, n_jobs=-1)
    selector.fit(x, y)
    return list(x.columns[selector.get_support()])

def fs_sfs(x, y, name):
    estimator = get_estimators('sfs')[name]
    sfs = SFS(estimator, k_features=k_features, forward=True, floating=False, scoring='roc_auc', cv=5, n_jobs=-1)
    sfs.fit(x, y)
    return list(sfs.k_feature_names_)

In [28]:
def fs_lasso(x, y):
    coef = Lasso(alpha=0.01).fit(x, y).coef_
    cols = list(x.columns[coef != 0])
    if len(cols) == 0:
        cols = list(x.columns[np.argsort(-np.abs(coef))[:k_features]])
    return cols

def fs_ridge(x, y):
    coef = np.abs(Ridge(alpha=0.3).fit(x, y).coef_)
    return list(x.columns[coef >= coef.mean()])

def fs_elastic_net(x, y):
    coef = ElasticNet(alpha=0.01, l1_ratio=0.5).fit(x, y).coef_
    cols = list(x.columns[coef != 0])
    if len(cols) == 0:
        cols = list(x.columns[np.argsort(-np.abs(coef))[:k_features]])
    return cols

In [29]:
def fs_boruta(x, y):
    rf = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5, random_state=42)
    boruta = BorutaPy(rf, n_estimators='auto', max_iter=100, random_state=42)
    boruta.fit(x.values, y.values)
    mask = boruta.support_ | boruta.support_weak_
    cols = list(x.columns[mask])
    if len(cols) == 0:
        cols = list(x.columns[np.argsort(boruta.ranking_)[:k_features]])
    return cols

def fs_stability(x, y, n_runs=100, threshold=0.6):
    counts = np.zeros(x.shape[1])
    for i in range(n_runs):
        idx = resample(np.arange(len(x)), n_samples=len(x) // 2, replace=False, stratify=y, random_state=i)
        model = LogisticRegression(penalty='l1', solver='liblinear', C=0.5).fit(x.iloc[idx], y.iloc[idx])
        counts += (model.coef_.ravel() != 0)
    freq = counts / n_runs
    cols = list(x.columns[freq >= threshold])
    if len(cols) == 0:
        cols = list(x.columns[np.argsort(-freq)[:k_features]])
    return cols

In [30]:
names10 = ['Random Forest', 'Logistic Regression', 'Decision Tree', 'XGBoost', 'CatBoost', 'AdaBoost', 'LightGBM', 'Support Vector Machine', 'Extra Trees', 'Gradient Boosting']

methods = [
    ('Correlation-based Feature Selection', lambda x, y: fs_correlation(x, y)),
    ('Chi-Square Feature Selection', lambda x, y: fs_chi_square(x, y)),
    ('Variance Threshold', lambda x, y: fs_variance(x, y)),
    ('Mann-Whitney U Test', lambda x, y: fs_mann_whitney(x, y)),
    ('Fisher Score', lambda x, y: fs_fisher(x, y)),
    ('Information Gain', lambda x, y: fs_information_gain(x, y)),
    ('Relief', lambda x, y: fs_relief(x, y)),
    ('Minimum Redundancy Maximum Relevance (mRMR)', lambda x, y: fs_mrmr(x, y)),
    ('Mutual Information', lambda x, y: fs_mutual_information(x, y)),
    ('ReliefF', lambda x, y: fs_relieff(x, y)),
]
methods += [('RFECV + ' + n, (lambda n: lambda x, y: fs_rfecv(x, y, n))(n)) for n in names10]
methods += [('SFS + ' + n, (lambda n: lambda x, y: fs_sfs(x, y, n))(n)) for n in names10]
methods += [
    ('Lasso (L1 Regularization)', lambda x, y: fs_lasso(x, y)),
    ('Ridge (L2 Regularization)', lambda x, y: fs_ridge(x, y)),
    ('Elastic Net', lambda x, y: fs_elastic_net(x, y)),
    ('Boruta Algorithm', lambda x, y: fs_boruta(x, y)),
    ('Stability Selection', lambda x, y: fs_stability(x, y)),
]
len(methods)

35

In [31]:
split_ratios = [0.8]

In [32]:
summary = []
for ratio in split_ratios:
    folder = '../Datasets/Split_' + str(round(ratio * 100)) + '/'
    train = pd.read_csv(folder + 'MinMax_train.csv')
    test = pd.read_csv(folder + 'MinMax_test.csv')
    x_train = train.drop('CKD', axis=1)
    y_train = train['CKD']
    x_test = test.drop('CKD', axis=1)
    y_test = test['CKD']
    for number, (name, function) in enumerate(methods, start=1):
        cols = function(x_train, y_train)
        df_train = x_train[cols].copy()
        df_train['label'] = y_train.values
        df_test = x_test[cols].copy()
        df_test['label'] = y_test.values
        df_train.to_csv(folder + 'Dataset_' + str(number) + '_train.csv', index=False)
        df_test.to_csv(folder + 'Dataset_' + str(number) + '_test.csv', index=False)
        summary.append({'Split Ratio': ratio, 'Dataset': 'Dataset_' + str(number), 'Method': name, 'Number of Features': len(cols), 'Features': ', '.join(cols)})
        print(ratio, number, name, len(cols))
summary_df = pd.DataFrame(summary)
summary_df.to_csv('../Datasets/Feature_Selection_Summary.csv', index=False)
summary_df

0.8 1 Correlation-based Feature Selection 21
0.8 2 Chi-Square Feature Selection 4
0.8 3 Variance Threshold 14
0.8 4 Mann-Whitney U Test 12
0.8 5 Fisher Score 11
0.8 6 Information Gain 11
0.8 7 Relief 11
0.8 8 Minimum Redundancy Maximum Relevance (mRMR) 11
0.8 9 Mutual Information 11
0.8 10 ReliefF 11
0.8 11 RFECV + Random Forest 8
0.8 12 RFECV + Logistic Regression 22
0.8 13 RFECV + Decision Tree 12
0.8 14 RFECV + XGBoost 20
0.8 15 RFECV + CatBoost 18
0.8 16 RFECV + AdaBoost 9
0.8 17 RFECV + LightGBM 5
0.8 18 RFECV + Support Vector Machine 22
0.8 19 RFECV + Extra Trees 6
0.8 20 RFECV + Gradient Boosting 19
0.8 21 SFS + Random Forest 11
0.8 22 SFS + Logistic Regression 11
0.8 23 SFS + Decision Tree 11
0.8 24 SFS + XGBoost 11
0.8 25 SFS + CatBoost 11
0.8 26 SFS + AdaBoost 11
0.8 27 SFS + LightGBM 11
0.8 28 SFS + Support Vector Machine 11
0.8 29 SFS + Extra Trees 11
0.8 30 SFS + Gradient Boosting 11
0.8 31 Lasso (L1 Regularization) 15
0.8 32 Ridge (L2 Regularization) 3
0.8 33 Elastic Net 

,Split Ratio,Dataset,Method,Number of Features,Features
0,0.8,Dataset_1,Correlation-based Feature Selection,21,"Gender, Job, Family_History, Height, Diabetic_..."
1,0.8,Dataset_2,Chi-Square Feature Selection,4,"Diabetic_Year, Heart_Disease, Urinary_Infectio..."
2,0.8,Dataset_3,Variance Threshold,14,"Gender, Family_History, Diet, Medicine, Insuli..."
3,0.8,Dataset_4,Mann-Whitney U Test,12,"Gender, Family_History, Diabetic_Year, Weight,..."
4,0.8,Dataset_5,Fisher Score,11,"Diabetic_Year, Heart_Disease, Pain_Killer, Ins..."
5,0.8,Dataset_6,Information Gain,11,"Diabetic_Year, Weight, Heart_Disease, Height, ..."
6,0.8,Dataset_7,Relief,11,"Insulin, Walk, Diabetic_Year, Medicine, Pain_K..."
7,0.8,Dataset_8,Minimum Redundancy Maximum Relevance (mRMR),11,"Diabetic_Year, Height, Heart_Disease, BMI, Die..."
8,0.8,Dataset_9,Mutual Information,11,"Diabetic_Year, Height, BMI, Heart_Disease, Wei..."
9,0.8,Dataset_10,ReliefF,11,"Diabetic_Year, Insulin, Hypertension, Zarda_Be..."
